# 01 — Introduction to Projective Geometric Algebra (PGA)

This notebook introduces **Projective Geometric Algebra (PGA)** — an extension of GA that unifies points, lines, and planes in a single framework. PGA is particularly powerful for robotics because it naturally represents rigid body motions.

## Learning Objectives

- Understand why PGA is useful for geometry and robotics
- Explain the role of the null basis e0 (homogenization)
- Compare VGA and PGA signatures
- Construct PGA2d and PGA3d algebras in AMSA

In [ ]:
# Setup
from amsa import Algebra
import numpy as np
import matplotlib.pyplot as plt

alg_2d = Algebra.pga2d()
alg_3d = Algebra.pga3d()

## 1.1 The Motivation for PGA

In VGA, we can represent points and vectors, but:

- **Lines** require bivectors (in 2D) or more complex constructions
- **Translations** are not naturally represented — they require limits of rotations
- **Infinity** is problematic — parallel lines don't meet

PGA solves this by adding a **homogeneous dimension** (the null basis e0). This unifies:

| VGA | PGA | What it represents |
|-----|-----|-------------------|
 | Vector | Line (bivector) | Direction + position |
 | — | Point (vector) | Homogeneous coordinates |
 | — | Motor | Translation + rotation |

As Charles Gunn (SIGGRAPH 2019) puts it: *PGA provides a coordinate-free framework for doing Euclidean geometry.*

## 1.2 PGA Signatures

PGA uses a degenerate metric — one dimension is nilpotent (squares to zero):

- **PGA2d**: signature (1, 1, 0) — two Euclidean + one null dimension
- **PGA3d**: signature (1, 1, 1, 0) — three Euclidean + one null dimension

The null basis `e0` satisfies $e_0^2 = 0$, which enables translations.

In [ ]:
print("=== PGA2d ===")
print("Dimension:", alg_2d.spec.dimension)
print("Signature:", alg_2d.spec.signature)
print("Blade count:", alg_2d.spec.blade_count)
print("\n=== PGA3d ===")
print("Dimension:", alg_3d.spec.dimension)
print("Signature:", alg_3d.spec.signature)
print("Blade count:", alg_3d.spec.blade_count)

## 1.3 PGA2d Blade Overview

PGA2d has 8 blades (2³ + 2¹ from the degenerate dimension). Let's examine them.

In [ ]:
print("PGA2d blades:")
for blade_index in range(alg_2d.spec.blade_count):
    name = alg_2d.spec.blade_name(blade_index)
    grade = alg_2d.spec.grade_of_blade(blade_index)
    print(f"  Blade {blade_index}: '{name}' (grade {grade})")

## 1.4 Understanding the Null Basis e0

The key insight: in PGA, we add a dimension with signature 0 (null):

- $e_0 \cdot e_0 = 0$ (nilpotent)
- This enables the homogenization of points

A point at $(x, y)$ in classical coordinates becomes $(x, y, 1)$ in PGA — the '1' is the e0 component.

In [ ]:
# In PGA2d, a point is a vector with e0 = 1
point = alg_2d.vector([1.0, 2.0, 1.0])  # x*e1 + y*e2 + 1*e0

print("Point (1, 2):", point.values)
print("Components:")
print("  e1 (x):", point.component("e1"))
print("  e2 (y):", point.component("e2"))
print("  e0 (homogeneous):", point.component("e0"))

# At infinity (parallel lines meet here)
point_inf = alg_2d.vector([1.0, 0.0, 0.0])  # e0 = 0 → point at infinity
print("\nPoint at infinity:", point_inf.values)
print("  (direction only, no position)")

## 1.5 Lines in PGA2d

Lines in PGA2d are bivectors (grade 2). A line can be written as:

$$L = a \cdot e_1 + b \cdot e_2 + c \cdot e_0$$

where $ax + by + c = 0$ is the classical line equation.

In [ ]:
# Define a line: x + y + 1 = 0 (passes through points where x+y=-1)
line = alg_2d.multivector({
    "e1": 1.0,  # coefficient of x
    "e2": 1.0,  # coefficient of y  
    "e0": 1.0   # constant term
})

print("Line (e1 + e2 + e0):", line.values)

# Check if points lie on the line using inner product
p1 = alg_2d.vector([0.0, -1.0, 1.0])  # (0, -1)
p2 = alg_2d.vector([1.0, -2.0, 1.0])  # (1, -2)

on_line_1 = p1 | line
on_line_2 = p2 | line

print("\nPoint (0, -1) on line:", on_line_1.component("e") == 0)
print("Point (1, -2) on line:", on_line_2.component("e") == 0)

## 1.6 Visualizing PGA2d

Let's visualize points and lines in PGA2d.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

# Draw coordinate axes
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)

# Points
points = [
    alg_2d.vector([1.0, 2.0, 1.0]),
    alg_2d.vector([-1.0, 1.0, 1.0]),
    alg_2d.vector([2.0, -1.0, 1.0])
]

for i, p in enumerate(points):
    x = p.component("e1") / p.component("e0")
    y = p.component("e2") / p.component("e0")
    ax.scatter(x, y, s=100, zorder=5)
    ax.text(x + 0.1, y + 0.1, f'P{i+1}', fontsize=10)

# Line: x + y + 1 = 0
x_vals = np.linspace(-3, 2, 100)
y_vals = -x_vals - 1
ax.plot(x_vals, y_vals, 'b-', linewidth=2, label='Line: x + y + 1 = 0')

ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.set_aspect('equal')
ax.set_title('PGA2d: Points and Lines', fontsize=12)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

## 1.7 Why PGA Matters for Robotics

PGA provides a unified framework for:

1. **Rigid body motions**: Motors (translations + rotations)
2. **Lines and points**: Natural intersection and join operations
3. **Projective duality**: Points ↔ Lines, planes ↔ points
4. **Kinematics**: Single algebra for forward and inverse kinematics
5. **Computer vision**: Homogeneous coordinates are standard

As noted in the original PGA paper (Gunn, arXiv:1901.05873):
> *PGA solves the dual problems of kinematics and geometry in a unified, coordinate-free way.*

## 1.8 Summary

We covered:

- **PGA motivation**: Unifies points, lines, translations in one algebra
- **Null basis e0**: Enables homogenization, $e_0^2 = 0$
- **PGA2d**: Signature (1, 1, 0), 8 blades
- **PGA3d**: Signature (1, 1, 1, 0), 16 blades
- **Points**: Vectors with e0 = 1
- **Lines**: Bivectors (grade 2)
- **Robotics**: Motors for rigid body motion

In the next notebook, we'll explore lines, points, and their meet/join operations.

## Exercises

### ⭐ Easy

**1.1** Create a point at (3, 4) in PGA2d and verify its homogeneous coordinate (e0) is 1.

In [ ]:
# Your turn: ⭐ Exercise 1.1
point = alg_2d.vector([3.0, 4.0, 1.0])
# TODO: Verify components
raise NotImplementedError("Implement exercise 1.1")

### ⭐⭐ Medium

**1.2** Create two parallel lines in PGA2d: L1: x = 0 and L2: x = 2. Compute their meet (intersection) — what happens? What does this tell you about PGA and parallel lines?

In [ ]:
# Your turn: ⭐⭐ Exercise 1.2
# L1: x = 0 → e1 component with e0 = 0
# L2: x = 2 → 
# TODO: Compute meet of parallel lines
raise NotImplementedError("Implement exercise 1.2")

### ⭐⭐⭐ Challenge

**1.3** Write a function that converts a classical line ax + by + c = 0 into a PGA2d bivector. Verify it works for at least 3 different lines.

In [ ]:
# Your turn: ⭐⭐⭐ Exercise 1.3
def classical_line_to_pga(a, b, c):
    """Convert ax + by + c = 0 to PGA2d bivector."""
    # TODO: Implement
raise NotImplementedError("Implement classical_line_to_pga function")

# Test cases
test_lines = [(1, 1, 1), (2, -1, 0), (0, 1, -3)]
for a, b, c in test_lines:
    line = classical_line_to_pga(a, b, c)
    print(f"Line {a}x + {b}y + {c} = 0:", line.values)

## Attribution

This notebook draws on:

- **Projective Geometric Algebra** — Charles G. Gunn
  https://arxiv.org/abs/1901.05873
- **SIGGRAPH 2019 Course Notes** — Charles G. Gunn
  https://arxiv.org/abs/2002.04509
- **PGABLE Tutorial** — Zachary Leger and Stephen Mann
  https://cs.uwaterloo.ca/~smann/PGABLE/PGAtutorial.pdf
- **Geometric Algebra for Computer Graphics** — John Vince
  https://link.springer.com/book/10.1007/978-1-84628-997-2